In [11]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import PowerTransformer


Question 1: Custom Feature Transformation Using FunctionTransformer

A. Apply log transformation using FunctionTransformer

In [12]:
df = pd.DataFrame({
    "Customer_ID": range(1, 11),
    "Income": [
        15000, 18000, 22000, 25000, 30000,
        35000, 42000, 55000, 120000, 500000
    ],
    "Monthly_Expenses": [
        8000, 9000, 11000, 12000, 15000,
        17000, 20000, 25000, 45000, 180000
    ]
})

# Select numerical features (exclude Customer_ID)
features = ["Income", "Monthly_Expenses"]

# Create FunctionTransformer
transformer = FunctionTransformer(np.log1p)

# Apply transformation
df_transformed = df.copy()

df_transformed[features] = transformer.transform(
    df[features]
)

print("Transformed DataFrame:")
print(df_transformed)

# Compare statistics
for col in features:
    original = df[col]
    transformed = df_transformed[col]

    print(f"\n{col}")
    print("Original mean:", round(original.mean(), 2))
    print("Transformed mean:", round(transformed.mean(), 4))

    print("Original range:",
          round(original.min(), 2),
          "to", round(original.max(), 2))

    print("Transformed range:",
          round(transformed.min(), 4),
          "to", round(transformed.max(), 4))

    print("Original std:", round(original.std(), 2))
    print("Transformed std:", round(transformed.std(), 4))


Transformed DataFrame:
   Customer_ID     Income  Monthly_Expenses
0            1   9.615872          8.987322
1            2   9.798183          9.105091
2            3   9.998843          9.305741
3            4  10.126671          9.392745
4            5  10.308986          9.615872
5            6  10.463132          9.741027
6            7  10.645449          9.903538
7            8  10.915107         10.126671
8            9  11.695255         10.714440
9           10  13.122365         12.100718

Income
Original mean: 86200.0
Transformed mean: 10.669
Original range: 15000 to 500000
Transformed range: 9.6159 to 13.1224
Original std: 148573.81
Transformed std: 1.0497

Monthly_Expenses
Original mean: 34200.0
Transformed mean: 9.8993
Original range: 8000 to 180000
Transformed range: 8.9873 to 12.1007
Original std: 52363.69
Transformed std: 0.9273


B. Build a pipeline and compare regression performance

In [13]:
X = df[["Income"]]
y = df["Monthly_Expenses"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42
)

# Model 1: Original Income
model_original = Pipeline([
    ("regressor", LinearRegression())
])

# Model 2: Log-transformed Income
model_log = Pipeline([
    ("log_transform", FunctionTransformer(np.log1p)),
    ("regressor", LinearRegression())
])

# Train both models
model_original.fit(X_train, y_train)
model_log.fit(X_train, y_train)

# Predictions
pred_original = model_original.predict(X_test)
pred_log = model_log.predict(X_test)

# Evaluation function
def evaluate(y_true, predictions, name):
    mae = mean_absolute_error(y_true, predictions)
    rmse = root_mean_squared_error(y_true, predictions)

    print(f"\n{name}")
    print("MAE:", round(mae, 2))
    print("RMSE:", round(rmse, 2))

evaluate(y_test, pred_original, "Original Income Model")
evaluate(y_test, pred_log, "Log Income Model")

# Compare actual and predicted values
comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Original_Prediction": pred_original,
    "Log_Prediction": pred_log
})

print("\nPrediction Comparison:")
print(comparison.round(2))



Original Income Model
MAE: 1108.23
RMSE: 1158.97

Log Income Model
MAE: 24318.31
RMSE: 29015.14

Prediction Comparison:
   Actual  Original_Prediction  Log_Prediction
0   45000             46313.41        91507.24
1    9000             10381.13        -6770.61
2   17000             16369.85        27677.08


Question 2: Designing and Validating Custom Transformations

A. Custom transformation using FunctionTransformer

In [14]:
df = pd.DataFrame({"Sales": [100, 150, 200, 250, 400, 600, 900, 1500, 3000, 8000],
    "Transaction_Count": [1, 2, 2, 3, 4, 5, 7, 10, 15, 30],
    "Discount_Percent": [0, 5, 10, 15, 20, 5, 10, 25, 30, 40],
    "Revenue": [120, 170, 230, 280, 450, 650, 950, 1600, 3200, 8500]})

# Custom function: Sales in thousands
def sales_in_thousands(x):
    return x / 1000

# Create transformers
sales_transformer = FunctionTransformer(sales_in_thousands)
sqrt_transformer = FunctionTransformer(np.sqrt)

# Apply transformations
df["Sales_Thousands"] = sales_transformer.transform(df[["Sales"]])

df["Transaction_Sqrt"] = sqrt_transformer.transform(
    df[["Transaction_Count"]]
)

print(df.round(3))


   Sales  Transaction_Count  Discount_Percent  Revenue  Sales_Thousands  \
0    100                  1                 0      120             0.10   
1    150                  2                 5      170             0.15   
2    200                  2                10      230             0.20   
3    250                  3                15      280             0.25   
4    400                  4                20      450             0.40   
5    600                  5                 5      650             0.60   
6    900                  7                10      950             0.90   
7   1500                 10                25     1600             1.50   
8   3000                 15                30     3200             3.00   
9   8000                 30                40     8500             8.00   

   Transaction_Sqrt  
0             1.000  
1             1.414  
2             1.414  
3             1.732  
4             2.000  
5             2.236  
6             2.646 

B. Custom clipping transformation

In [15]:
# Reusable clipping function
def clip_values(X, lower, upper):
    return np.clip(X, lower, upper)

# Two clipping ranges
clip_1 = FunctionTransformer(
    clip_values,
    kw_args={"lower": 100, "upper": 3000}
)

clip_2 = FunctionTransformer(
    clip_values,
    kw_args={"lower": 100, "upper": 1500}
)

# Apply clipping
df["Sales_Clip_3000"] = clip_1.transform(df[["Sales"]])
df["Sales_Clip_1500"] = clip_2.transform(df[["Sales"]])

print(df[["Sales","Sales_Clip_3000","Sales_Clip_1500"]])

# Compare statistics
def compare_stats(series, name):
    print(f"\n{name}")
    print("Mean:", round(series.mean(), 2))
    print("Std:", round(series.std(), 2))
    print("Min:", series.min())
    print("Max:", series.max())

compare_stats(df["Sales"], "Original Sales")
compare_stats(df["Sales_Clip_3000"], "Clipped Sales (100, 3000)")
compare_stats(df["Sales_Clip_1500"], "Clipped Sales (100, 1500)")


   Sales  Sales_Clip_3000  Sales_Clip_1500
0    100              100              100
1    150              150              150
2    200              200              200
3    250              250              250
4    400              400              400
5    600              600              600
6    900              900              900
7   1500             1500             1500
8   3000             3000             1500
9   8000             3000             1500

Original Sales
Mean: 1510.0
Std: 2447.54
Min: 100
Max: 8000

Clipped Sales (100, 3000)
Mean: 1010.0
Std: 1131.08
Min: 100
Max: 3000

Clipped Sales (100, 1500)
Mean: 710.0
Std: 593.86
Min: 100
Max: 1500


Question 3: PowerTransformer and Distribution Analysis

A. Apply Yeo-Johnson transformation and compare skewness

In [16]:
df = pd.DataFrame({"Annual_Income": [20000, 22000, 25000, 28000, 32000,40000, 50000, 70000, 150000, 600000],
    "Loan_Amount": [5000, 7000, 9000, 12000, 15000, 20000, 30000, 50000, 100000, 400000],
    "Account_Balance": [-500, 0, 200, 500, 1000, 2500, 4000, 8000, 15000, 50000]})

# Select numerical features
features = ["Annual_Income","Loan_Amount","Account_Balance"]

# Original skewness
skew_before = df[features].skew()

# Apply Yeo-Johnson
pt = PowerTransformer(method="yeo-johnson")

transformed = pt.fit_transform(df[features])

df_transformed = pd.DataFrame(transformed,columns=features)

# Transformed skewness
skew_after = df_transformed.skew()

# Compare skewness
comparison = pd.DataFrame({"Skewness_Before": skew_before,"Skewness_After": skew_after})

print("Skewness Comparison:")
print(comparison.round(4))

# Learned lambda values
lambda_values = pd.Series(pt.lambdas_,index=features,name="Lambda")

print("\nLearned Lambda Values:")
print(lambda_values.round(4))

print("\nTransformed Data:")
print(df_transformed.round(3))


Skewness Comparison:
                 Skewness_Before  Skewness_After
Annual_Income             2.9096          0.4283
Loan_Amount               2.8604          0.1895
Account_Balance           2.6636          0.9304

Learned Lambda Values:
Annual_Income     -0.7816
Loan_Amount       -0.3554
Account_Balance    0.7416
Name: Lambda, dtype: float64

Transformed Data:
   Annual_Income  Loan_Amount  Account_Balance
0         -1.380       -1.599           -1.791
1         -1.135       -1.119           -0.435
2         -0.834       -0.796           -0.389
3         -0.591       -0.460           -0.344
4         -0.332       -0.223           -0.282
5          0.046        0.057           -0.131
6          0.364        0.406           -0.004
7          0.749        0.780            0.286
8          1.323        1.190            0.715
9          1.790        1.762            2.376


B. Apply Box-Cox and implement safe preprocessing

In [17]:
# Box-Cox for positive features
positive_features = ["Annual_Income", "Loan_Amount"]

boxcox_transformers = {}
boxcox_results = {}

for col in positive_features:
    transformer = PowerTransformer(method="box-cox")

    boxcox_results[col] = transformer.fit_transform(
        df[[col]]
    ).ravel()

    boxcox_transformers[col] = transformer

    print(f"\n{col}")
    print("Lambda:", round(transformer.lambdas_[0], 4))
    print("Transformed:", np.round(
        boxcox_results[col], 3
    ))

# Attempt Box-Cox on Account_Balance
try:
    boxcox_balance = PowerTransformer(method="box-cox")
    boxcox_balance.fit_transform(df[["Account_Balance"]])
except ValueError as e:
    print("\nBox-Cox Error:")
    print(e)



Annual_Income
Lambda: -0.7816
Transformed: [-1.38  -1.135 -0.834 -0.591 -0.332  0.046  0.364  0.749  1.323  1.79 ]

Loan_Amount
Lambda: -0.3554
Transformed: [-1.599 -1.119 -0.796 -0.46  -0.223  0.057  0.406  0.78   1.19   1.762]

Box-Cox Error:
The Box-Cox transformation can only be applied to strictly positive data
